In [ ]:
# load the correlation results
from general_utils import load_temporary_data

# path that you passed as   save_folder / (save_name + ".csv")
zarr_path = "/root/capsule/scratch/correlation_results/sig_dir_all_sessions.zarr"
ds = load_temporary_data(zarr_path)

print("Rows:", len(ds))
display(ds.head())             # in a notebook


In [ ]:
import importlib
import correlation_utils   # or the module where the function lives

importlib.reload(correlation_utils)

from correlation_utils import get_column_names

get_column_names(ds)

In [ ]:

import importlib
import correlation_utils   # or the module where the function lives

importlib.reload(correlation_utils)

from correlation_utils import select_units_by_significance


selected_based_on_sumQ, _, result_dicts,output_dicts_based_on_sumQ = select_units_by_significance(
    ds,
    pval_col="simple_LR-QLearning_L2F1_softmax-sumQ-1-g1-s0-d0-pval",
    coef_col="simple_LR-QLearning_L2F1_softmax-sumQ-1-g1-s0-d0-coef",
    time_window="-1_0",
    alpha=0.05,
    brain_areas=["SI","MA"],
    coef_col_sign=("positive", "negative"),
    output_time_window=["0.3_2","-1_0","0.3_2","-1_0"],
    output_columns=["simple_LR-QLearning_L2F1_softmax-reward-g1-s0-d0-coef","simple_LR-QLearning_L2F1_softmax-sumQ-1-g1-s0-d0-coef","simple_LR-QLearning_L2F1_softmax-reward-g1-s0-d0-pval","simple_LR-QLearning_L2F1_softmax-sumQ-1-g1-s0-d0-pval"]
)



selected_based_on_reward, _, result_dicts,output_dicts_based_on_reward = select_units_by_significance(
    ds,
    pval_col="simple_LR-QLearning_L2F1_softmax-reward-g1-s0-d0-pval",
    coef_col="simple_LR-QLearning_L2F1_softmax-reward-g1-s0-d0-coef",
    time_window="0.3_2",
    alpha=0.05,
    brain_areas=["SI","MA"],
    coef_col_sign=("positive", "negative"),
    output_time_window=["0.3_2","-1_0","0.3_2","-1_0"],
    output_columns=["simple_LR-QLearning_L2F1_softmax-reward-g1-s0-d0-coef","simple_LR-QLearning_L2F1_softmax-sumQ-1-g1-s0-d0-coef","simple_LR-QLearning_L2F1_softmax-reward-g1-s0-d0-pval","simple_LR-QLearning_L2F1_softmax-sumQ-1-g1-s0-d0-pval"]
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# ---------------------------------------------------------
# 1) Convert each selected list to a set of tuples
# ---------------------------------------------------------

set_sumQ = set(zip(
    selected_based_on_sumQ["session_id"],
    selected_based_on_sumQ["unit_index"]
))

set_reward = set(zip(
    selected_based_on_reward["session_id"],
    selected_based_on_reward["unit_index"]
))

# ---------------------------------------------------------
# 2) Compute overlap summary
# ---------------------------------------------------------

only_sumQ = len(set_sumQ - set_reward)
only_reward = len(set_reward - set_sumQ)
overlap = len(set_sumQ & set_reward)

print("\n=== Summary ===")
print(f"SumQ only:   {only_sumQ}")
print(f"Reward only: {only_reward}")
print(f"Overlap:     {overlap}")

# ---------------------------------------------------------
# 3) Plot Venn Diagram
# ---------------------------------------------------------

plt.figure(figsize=(6, 6))
venn2(
    subsets = (only_sumQ, only_reward, overlap),
    set_labels = ("SumQ selected", "Reward selected")
)
plt.title("Overlap of Units Selected by SumQ vs Reward")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 1) Convert output_dicts to DataFrame and extract columns
# =========================================================

# output_dicts is assumed to be a list of dicts already defined
df = pd.DataFrame(output_dicts)

reward_col = "simple_LR-QLearning_L2F1_softmax-reward-g1-s0-d0-coef"
sumQ_col   = "simple_LR-QLearning_L2F1_softmax-sumQ-1-g1-s0-d0-coef"

reward_coefs = df[reward_col]
sumQ_coefs   = df[sumQ_col]

df_plot = pd.DataFrame({
    "reward": reward_coefs,
    "sumQ": sumQ_coefs,
})

# =========================================================
# 2) Compute quadrants for each point
# =========================================================

def get_quadrant(row):
    """
    Assign a quadrant label based on the sign of reward and sumQ.

    Q1 (+,+): reward > 0, sumQ > 0
    Q2 (-,+): reward < 0, sumQ > 0
    Q3 (-,-): reward < 0, sumQ < 0
    Q4 (+,-): reward > 0, sumQ < 0
    On Axis : any point where reward == 0 or sumQ == 0
    """
    if row["reward"] > 0 and row["sumQ"] > 0:
        return "Q1 (+,+)"
    elif row["reward"] < 0 and row["sumQ"] > 0:
        return "Q2 (-,+)"
    elif row["reward"] < 0 and row["sumQ"] < 0:
        return "Q3 (-,-)"
    elif row["reward"] > 0 and row["sumQ"] < 0:
        return "Q4 (+,-)"
    else:
        return "On Axis"

df_plot["quadrant"] = df_plot.apply(get_quadrant, axis=1)

# =========================================================
# 3) Compute counts and fractions per quadrant
# =========================================================

counts = df_plot["quadrant"].value_counts()
fractions = counts / len(df_plot)

summary_table = pd.DataFrame({
    "count": counts,
    "fraction": fractions.round(3),
})

print("Quadrant Summary:")
print(summary_table)

# =========================================================
# 4) Scatter plot with fitted line
# =========================================================

x = df_plot["reward"].values
y = df_plot["sumQ"].values

# Fit a linear regression line y = a*x + b
a, b = np.polyfit(x, y, 1)
print(f"Slope (a): {a}")
print(f"Intercept (b): {b}")

# Generate line for plotting
x_line = np.linspace(x.min(), x.max(), 100)
y_line = a * x_line + b

plt.figure(figsize=(6, 6))

# Scatter points
plt.scatter(x, y, alpha=0.6, s=20, label="Units")

# Fitted line
plt.plot(x_line, y_line, linewidth=2, label=f"Fit: y = {a:.2f}x + {b:.2f}")

# Zero axes for quadrant reference
plt.axhline(0, color="gray", linewidth=1)
plt.axvline(0, color="gray", linewidth=1)

plt.xlabel("Reward Coefficient")
plt.ylabel("SumQ Coefficient")
plt.title("Reward vs SumQ Coefficients with Linear Fit")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# =========================================================
# 5) Bar plot of quadrant fractions
# =========================================================

plt.figure(figsize=(6, 4))
plt.bar(summary_table.index, summary_table["fraction"])

plt.ylabel("Fraction of Units")
plt.title("Fraction of Units in Each Quadrant")
plt.xticks(rotation=30)
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
